# 02 — Mention Extraction, Validation, and Normalization

This notebook validates the extraction pipeline (`src/mention_extractor.py`) and scientific text normalization (`src/preprocessing.py`).

Important dataset finding documented here:
- On 520 of 521 topics, `flatten_mentions` on `flatten_tokens` matches document-level `mentions` on `tokens[snippet_idx]` identically.
- On Test topic `id=1034`, `flatten_mentions` is misaligned (including 1 out-of-bounds span `[3972, 3975, 6]` for `len(flatten_tokens)=3963`). Our extractor detects this mismatch and recovers all 62 affected mentions cleanly from `tokens[snippet_idx]` without dropping records.


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)
print("Project root:", ROOT)


Project root: E:\AOU\elevvo internship\tusk 4


In [2]:
from src.data_loader import load_config, load_all_splits
from src.mention_extractor import extract_mentions_from_dataframe
from src.preprocessing import normalize_mention_text

cfg = load_config(ROOT / "config.yaml")
splits = load_all_splits(cfg)

reports = []
extracted = {}
for sname, df in splits.items():
    mdf, st = extract_mentions_from_dataframe(
        df,
        sname,
        span_end_inclusive=cfg["extraction"]["span_end_inclusive"],
        prefer_doc_tokens_on_mismatch=cfg["extraction"]["prefer_doc_tokens_on_mismatch"],
        preprocessing_cfg=cfg["preprocessing"],
    )
    extracted[sname] = mdf
    reports.append(st.to_dict())

display(pd.DataFrame(reports).drop(columns=["anomalies_sample"]))
print("Sample Test anomalies recovered via doc_tokens_fallback:")
display(pd.DataFrame(reports[-1]["anomalies_sample"][:8]))


Split 'test' extraction completed: 10423 extracted, 62 doc_tokens_fallback, 0 malformed.


,split,total_topics,total_raw_mentions,extracted_mentions,flatten_aligned_mentions,doc_fallback_mentions,malformed_mentions,empty_normalized_mentions,anomaly_count
0,train,221,10922,10922,10922,0,0,0,0
1,validation,100,4873,4873,4873,0,0,0,0
2,test,200,10423,10423,10361,62,0,0,62


Sample Test anomalies recovered via doc_tokens_fallback:


,topic_id,mention_idx,reason
0,1034,0,Span mismatch flatten='collect terms with' vs doc='text classification task'...
1,1034,1,Span mismatch flatten='social networks from' vs doc='text classification pro...
2,1034,2,Span mismatch flatten='not applicable to' vs doc='text classification proble...
3,1034,3,Span mismatch flatten='to resolve' vs doc='text-based classification'; using...
4,1034,4,Span mismatch flatten='of different' vs doc='text-based classification'; usi...
5,1034,5,Span mismatch flatten='of documents' vs doc='text classifying'; using doc_to...
6,1034,6,"Span mismatch flatten='troubleshooting , and' vs doc='text data classificati..."
7,1034,7,Span mismatch flatten='of text image' vs doc='text data classification'; usi...


In [3]:
# Verify topic-scoped cluster keys prevent cross-topic collisions
for sname, mdf in extracted.items():
    raw_unique = mdf["raw_cluster_id"].nunique()
    scoped_unique = mdf["cluster_id"].nunique()
    print(f"{sname}: raw_cluster_id unique={raw_unique} vs scoped cluster_id unique={scoped_unique}")


train: raw_cluster_id unique=45 vs scoped cluster_id unique=4079
validation: raw_cluster_id unique=41 vs scoped cluster_id unique=1747
test: raw_cluster_id unique=46 vs scoped cluster_id unique=3711


In [4]:
# Normalization unit examples
examples = [
    "BERT",
    "bert",
    " BERT ",
    "Bidirectional Encoder Representations from Transformers",
    "Rectified Linear Units ( ReLU )",
    "ResNet-50",
    "C++",
    "Self\u2013Attention , ",
]
norm_df = pd.DataFrame({
    "original": examples,
    "normalized": [normalize_mention_text(x) for x in examples],
})
display(norm_df)


,original,normalized
0,BERT,bert
1,bert,bert
2,BERT,bert
3,Bidirectional Encoder Representations from Transformers,bidirectional encoder representations from transformers
4,Rectified Linear Units ( ReLU ),rectified linear units ( relu )
5,ResNet-50,resnet-50
6,C++,c++
7,"Self–Attention ,",self-attention
